# Chapter 11 -- Execution Graphs & Agent Runtimes (Solved)

Work through this notebook **after reading** `notes/ch11-execution-graphs.md`. Neither LangGraph nor the Claude Agent SDK is installed in this environment, so this notebook does what this book has done from Chapter 2 onward: build the real primitive by hand first. Part 1 implements the same 5-node agentic-RAG task from notes Section 11 as a **hand-rolled loop** (Chapter 6's style). Part 2 implements a genuine, small **graph engine from scratch** -- real nodes, edges, conditional edges, reducers, and checkpointing, not a simulation of one -- and runs the identical task through it, reproducing the notes' exact state trace. The two implementations are then compared on real, measured lines of code.

Three exercises below have a stub to fill in: **summarization middleware** (notes Section 4, tied back to Chapter 4's compaction), **permission middleware** (notes Section 4, tied to Chapter 16), and **an interrupt-and-resume flow** (notes Section 6). Everything is fully offline and deterministic -- no API key needed for any exercise.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import anthropic


def _find_and_load_env() -> None:
    """Walk up from the current working directory to find and load a repo-root .env file, if one exists."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        candidate = parent / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    print("No .env file found -- copy .env.example to .env at the repo root to enable the real-model sections.")


_find_and_load_env()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_NAME = os.getenv("BEDROCK_MODEL_ID", "anthropic.claude-sonnet-5")


def test_connection(client, model_name):
    """Send a trivial ping to confirm the Bedrock connection actually works."""
    print(f"Testing connection to Bedrock (model={model_name})...")
    try:
        response = client.messages.create(
            model=model_name, max_tokens=10,
            messages=[{"role": "user", "content": "Reply with exactly the word: pong"}],
        )
        reply = next((b.text for b in response.content if b.type == "text"), "")
        status = "PASS" if "pong" in reply.lower() else f"unexpected reply: {reply!r}"
        print(f"  {status}")
    except Exception as exc:
        print(f"  Connection check FAILED: {type(exc).__name__}: {exc}")
        print()
        print("The rest of this notebook still works fully offline -- this cell")
        print("only matters for the optional real-model section at the end.")


if not AWS_ACCESS_KEY_ID or not AWS_SECRET_ACCESS_KEY:
    print("AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY not set in .env -- skipping connection test.")
    print("The rest of this notebook still works fully offline.")
else:
    client = anthropic.AnthropicBedrockMantle(
        aws_access_key=AWS_ACCESS_KEY_ID, aws_secret_key=AWS_SECRET_ACCESS_KEY, aws_region=AWS_REGION,
    )
    test_connection(client, MODEL_NAME)


## Part 1 -- The Task, Hand-Rolled as a Plain Loop (Given)

The exact same task as notes Section 11: retrieve, critique, and -- if the evidence is insufficient -- rewrite the query and retrieve again, before generating a final answer. Written as a Chapter-6-style `while`/`if` loop, with no graph structure at all: the possible sequence of steps lives entirely in this function's own control flow.

In [ ]:
def run_hand_rolled_loop(query):
    """Chapter 6 style: a plain loop, if/else branching, no pre-declared graph structure."""
    messages = [f"user: {query}"]
    retrieved_docs = []
    confidence = 0.0
    current_step = "start"

    messages.append("assistant: retrieved D1 (2023 pricing doc)")
    retrieved_docs.append("D1")
    confidence = 0.3
    current_step = "retrieve1"
    print(f"  [loop] step={current_step:10s} confidence={confidence}")

    messages.append("assistant: D1 covers 2023, not 2024 -- insufficient")
    current_step = "critique"
    print(f"  [loop] step={current_step:10s} confidence={confidence}")

    if confidence < 0.8:
        messages.append("assistant: rewriting query -> '2024 pricing specifically'")
        current_step = "rewrite"
        print(f"  [loop] step={current_step:10s} confidence={confidence}")

        messages.append("assistant: retrieved D2 (2024 pricing doc)")
        retrieved_docs.append("D2")
        confidence = 0.9
        current_step = "retrieve2"
        print(f"  [loop] step={current_step:10s} confidence={confidence}")

    messages.append("assistant: 2024 pricing is $49/mo, per D2")
    current_step = "generate"
    print(f"  [loop] step={current_step:10s} confidence={confidence}")

    return {"messages": messages, "retrieved_docs": retrieved_docs, "confidence": confidence, "current_step": current_step}


print("-" * 60)
print("HAND-ROLLED LOOP")
print("-" * 60)
loop_result = run_hand_rolled_loop("what's the 2024 pricing?")
print(f"\nFinal state: {loop_result}")


## Part 2 -- A Real Graph Engine, Built From Scratch (Given)

Genuine nodes, fixed edges, conditional edges, per-field reducers, middleware wrapping, and checkpointing after every node -- notes Sections 2, 3, and 5, implemented rather than described. This is a minimal stand-in for what LangGraph provides as a library; the mechanism is real, just small.

In [ ]:
def append_reducer(old, new):
    """notes Section 3 -- combine, never replace. `old` may be None on the first write."""
    return (old or []) + list(new)


def overwrite_reducer(old, new):
    """notes Section 3 -- last-write-wins."""
    return new


class Graph:
    def __init__(self):
        self.nodes = {}
        self.edges = {}
        self.conditional_edges = {}
        self.reducers = {}

    def add_node(self, name, fn):
        self.nodes[name] = fn

    def add_edge(self, a, b):
        self.edges[a] = b

    def add_conditional_edge(self, a, condition_fn, mapping):
        self.conditional_edges[a] = (condition_fn, mapping)

    def set_reducer(self, key, fn):
        self.reducers[key] = fn


def run_steps(graph, start_node, state, middlewares=None):
    """
    Generator yielding ("before"|"after", node_name, state_snapshot) around
    every node execution -- the primitive both invoke() and the interrupt
    exercise are built on top of.
    """
    node = start_node
    while node is not None:
        yield ("before", node, dict(state))
        fn = graph.nodes[node]
        wrapped = fn
        for mw in reversed(middlewares or []):
            wrapped = mw(wrapped, node)
        update = wrapped(state) or {}
        for key, value in update.items():
            reducer = graph.reducers.get(key, overwrite_reducer)
            state[key] = reducer(state.get(key), value)
        yield ("after", node, dict(state))

        if node in graph.conditional_edges:
            condition_fn, mapping = graph.conditional_edges[node]
            node = mapping.get(condition_fn(state))
        else:
            node = graph.edges.get(node)


def invoke(graph, start_node, initial_state, middlewares=None):
    """Drain run_steps fully -- notes Section 5's checkpointer, one snapshot per node transition."""
    state = dict(initial_state)
    checkpoints = [dict(state)]
    for phase, node, snapshot in run_steps(graph, start_node, state, middlewares):
        if phase == "after":
            checkpoints.append(dict(state))
    return state, checkpoints


print("Graph engine defined: Graph, append_reducer, overwrite_reducer, run_steps, invoke.")


## Part 3 -- Running the Same Task Through the Graph, Verified Against Notes Section 11

Five nodes, two reducers, one conditional edge -- the exact scenario from the notes dry-run, now actually executed instead of hand-traced, with every intermediate state asserted against the notes' exact numbers.

In [ ]:
def node_retrieve1(state):
    return {"messages": ["assistant: retrieved D1 (2023 pricing doc)"], "retrieved_docs": ["D1"], "confidence": 0.3, "current_step": "retrieve1"}


def node_critique(state):
    return {"messages": ["assistant: D1 covers 2023, not 2024 -- insufficient"], "confidence": 0.3, "current_step": "critique"}


def node_rewrite(state):
    return {"messages": ["assistant: rewriting query -> '2024 pricing specifically'"], "current_step": "rewrite"}


def node_retrieve2(state):
    return {"messages": ["assistant: retrieved D2 (2024 pricing doc)"], "retrieved_docs": ["D2"], "confidence": 0.9, "current_step": "retrieve2"}


def node_generate(state):
    return {"messages": ["assistant: 2024 pricing is $49/mo, per D2"], "current_step": "generate"}


def route_after_critique(state):
    return "enough" if state["confidence"] >= 0.8 else "not_enough"


rag_graph = Graph()
rag_graph.add_node("retrieve1", node_retrieve1)
rag_graph.add_node("critique", node_critique)
rag_graph.add_node("rewrite", node_rewrite)
rag_graph.add_node("retrieve2", node_retrieve2)
rag_graph.add_node("generate", node_generate)
rag_graph.add_edge("retrieve1", "critique")
rag_graph.add_conditional_edge("critique", route_after_critique, {"enough": "generate", "not_enough": "rewrite"})
rag_graph.add_edge("rewrite", "retrieve2")
rag_graph.add_edge("retrieve2", "generate")
rag_graph.set_reducer("messages", append_reducer)
rag_graph.set_reducer("retrieved_docs", append_reducer)
rag_graph.set_reducer("confidence", overwrite_reducer)
rag_graph.set_reducer("current_step", overwrite_reducer)

initial_state = {"messages": ["user: what's the 2024 pricing?"], "retrieved_docs": [], "confidence": 0.0, "current_step": "start"}
final_state, checkpoints = invoke(rag_graph, "retrieve1", initial_state)

print("-" * 60)
print("GRAPH ENGINE: state after every checkpoint")
print("-" * 60)
for i, snap in enumerate(checkpoints):
    print(f"  checkpoint {i}: step={snap['current_step']:10s} confidence={snap['confidence']}  "
          f"messages={len(snap['messages'])}  docs={snap['retrieved_docs']}")

assert len(checkpoints) == 6, f"expected 6 checkpoints (initial + 5 nodes), got {len(checkpoints)}"
assert final_state["messages"] == loop_result["messages"], "graph and hand-rolled loop should reach identical messages"
assert final_state["retrieved_docs"] == ["D1", "D2"]
assert final_state["confidence"] == 0.9
assert final_state["current_step"] == "generate"
print("\nConfirmed: matches notes Section 11's hand-traced state exactly, and matches")
print("the hand-rolled loop's final result too -- same task, same answer, different structure.")


## Part 4 -- Middleware and Logging (Given)

A basic logging middleware, wrapping every node's execution -- notes Section 4's mechanism, applied once at the graph level rather than pasted into every node function.

In [ ]:
def logging_middleware(next_fn, node_name):
    """Wraps ANY node -- notes Section 4's whole point: written once, applied everywhere."""
    def wrapped(state):
        print(f"  [middleware] entering node '{node_name}'")
        result = next_fn(state)
        print(f"  [middleware] node '{node_name}' returned keys: {list(result.keys())}")
        return result
    return wrapped


print("-" * 60)
print("RUNNING WITH LOGGING MIDDLEWARE ATTACHED")
print("-" * 60)
final_state_2, _ = invoke(rag_graph, "retrieve1", dict(initial_state), middlewares=[logging_middleware])
assert final_state_2 == final_state, "middleware must not change the actual computed result, only observe it"
print("\nSame final state, with every node transition now logged -- the middleware added")
print("observability without touching a single node function's own code.")


## Exercise 1 -- Summarization Middleware

Implement `make_summarization_middleware(max_messages, keep_recent)`: notes Section 4, tied directly back to Chapter 4's compaction. It should return a middleware function that, **before** calling the wrapped node, checks whether `state["messages"]` has grown past `max_messages`; if so, replace it with `[a one-line summary of the older messages] + the last keep_recent messages`, mutating `state` directly. This is a deliberate, explicit override of the normal append reducer -- exactly the same "compaction is a deliberate exception to normal accumulation" idea Chapter 4 built, now expressed as middleware instead of an ad-hoc check inside the loop.

In [ ]:
def make_summarization_middleware(max_messages=4, keep_recent=2):
    """Factory: returns a middleware that compacts state['messages'] before a node runs, if it's grown too large."""
    def middleware(next_fn, node_name):
        def wrapped(state):
            messages = state.get("messages", [])
            if len(messages) > max_messages:
                older = messages[:-keep_recent]
                recent = messages[-keep_recent:]
                summary = f"[summary of {len(older)} earlier messages]"
                state["messages"] = [summary] + recent
                print(f"  [summarization middleware] compacted {len(messages)} -> {len(state['messages'])} messages before '{node_name}'")
            return next_fn(state)
        return wrapped
    return middleware


In [ ]:
summarization_mw = make_summarization_middleware(max_messages=3, keep_recent=2)

print("-" * 60)
print("RUNNING WITH SUMMARIZATION MIDDLEWARE (max_messages=3, keep_recent=2)")
print("-" * 60)
final_state_3, checkpoints_3 = invoke(rag_graph, "retrieve1", dict(initial_state), middlewares=[summarization_mw])

for i, snap in enumerate(checkpoints_3):
    print(f"  checkpoint {i}: {len(snap['messages'])} messages")

assert len(final_state_3["messages"]) < len(final_state["messages"]), "summarization should leave FEWER messages in the final state than the uncompacted run"
assert any(m.startswith("[summary of") for m in final_state_3["messages"]), "a summary marker should appear somewhere in the final messages"
assert final_state_3["confidence"] == final_state["confidence"] and final_state_3["retrieved_docs"] == final_state["retrieved_docs"]
print(f"\nExercise 1 PASSED -- final message count dropped from {len(final_state['messages'])} to")
print(f"{len(final_state_3['messages'])}, while confidence and retrieved_docs stayed identical --")
print("summarization only touches the field it's meant to touch.")


## Exercise 2 -- Permission Middleware

Implement `make_permission_middleware(destructive_nodes, approved)`: notes Section 4 (tied forward to Chapter 16). It should return a middleware that, when wrapping a node whose name is in `destructive_nodes` and NOT in `approved`, prints a rejection message and **raises `PermissionError(node_name)`** instead of calling the wrapped node at all. Nodes not in `destructive_nodes`, or that ARE in `approved`, should run normally.

In [ ]:
def make_permission_middleware(destructive_nodes, approved=None):
    """Factory: blocks any node in destructive_nodes unless it's also in approved."""
    approved = approved or set()

    def middleware(next_fn, node_name):
        def wrapped(state):
            if node_name in destructive_nodes and node_name not in approved:
                print(f"  [permission middleware] BLOCKED node '{node_name}' -- not approved")
                raise PermissionError(node_name)
            return next_fn(state)
        return wrapped
    return middleware


In [ ]:
print("-" * 60)
print("RUN 1: retrieve2 is destructive and NOT approved -- should be blocked")
print("-" * 60)
blocked_mw = make_permission_middleware(destructive_nodes={"retrieve2"}, approved=set())
try:
    invoke(rag_graph, "retrieve1", dict(initial_state), middlewares=[blocked_mw])
    raised = False
except PermissionError as exc:
    raised = True
    print(f"  Caught PermissionError({exc}) -- exactly as expected.")

assert raised, "the run should have been blocked with a PermissionError at retrieve2"

print()
print("-" * 60)
print("RUN 2: retrieve2 is destructive but APPROVED -- should complete normally")
print("-" * 60)
approved_mw = make_permission_middleware(destructive_nodes={"retrieve2"}, approved={"retrieve2"})
final_state_4, _ = invoke(rag_graph, "retrieve1", dict(initial_state), middlewares=[approved_mw])
assert final_state_4 == final_state, "an approved run should reach the identical final state as the unrestricted run"
print("\nExercise 2 PASSED -- the same node is blocked or allowed purely based on the")
print("approval set, with zero changes to the node's own function.")


## Exercise 3 -- Interrupt-and-Resume

Implement `invoke_with_interrupt(graph, start_node, initial_state, interrupt_before, middlewares=None)`: notes Section 6. Drive `run_steps` manually; the moment you see a `("before", node, snapshot)` event where `node == interrupt_before`, **stop without executing that node** and return `{"paused": True, "next_node": node, "state": <the state dict as of the pause>, "checkpoints": checkpoints}`. If the generator finishes without ever hitting `interrupt_before`, return `{"paused": False, "next_node": None, "state": state, "checkpoints": checkpoints}` instead.

In [ ]:
def invoke_with_interrupt(graph, start_node, initial_state, interrupt_before, middlewares=None):
    """Run until (but not including) interrupt_before, then pause -- notes Section 6."""
    state = dict(initial_state)
    checkpoints = [dict(state)]
    for phase, node, snapshot in run_steps(graph, start_node, state, middlewares):
        if phase == "before" and node == interrupt_before:
            return {"paused": True, "next_node": node, "state": dict(state), "checkpoints": checkpoints}
        if phase == "after":
            checkpoints.append(dict(state))
    return {"paused": False, "next_node": None, "state": state, "checkpoints": checkpoints}


def resume(graph, paused_result, edited_state=None, middlewares=None):
    """Continue a paused run from paused_result['next_node'], optionally with human-edited state."""
    state = edited_state if edited_state is not None else paused_result["state"]
    return invoke(graph, paused_result["next_node"], state, middlewares)


In [ ]:
print("-" * 60)
print("PAUSING before 'retrieve2' -- the notionally expensive/destructive step")
print("-" * 60)
paused = invoke_with_interrupt(rag_graph, "retrieve1", dict(initial_state), interrupt_before="retrieve2")

assert paused["paused"] is True
assert paused["next_node"] == "retrieve2"
print(f"Paused. next_node={paused['next_node']!r}")
print(f"State at pause: confidence={paused['state']['confidence']}, retrieved_docs={paused['state']['retrieved_docs']}")

print()
print("-" * 60)
print("A human edits state before resuming -- adding a manually-sourced doc")
print("-" * 60)
edited_state = dict(paused["state"])
edited_state["retrieved_docs"] = list(edited_state["retrieved_docs"]) + ["human-provided-context"]
print(f"Edited retrieved_docs: {edited_state['retrieved_docs']}")

final_state_5, checkpoints_5 = resume(rag_graph, paused, edited_state=edited_state)

print(f"\nFinal retrieved_docs after resume: {final_state_5['retrieved_docs']}")
assert final_state_5["retrieved_docs"] == ["D1", "human-provided-context", "D2"], \
    f"expected the human's edit to survive alongside D1 and D2, got {final_state_5['retrieved_docs']}"
assert final_state_5["confidence"] == 0.9 and final_state_5["current_step"] == "generate"
print("\nExercise 3 PASSED -- the run paused with real, inspectable state; the human's")
print("edit survived resumption (append reducer preserved it, exactly as D1 and D2 were),")
print("and the graph correctly finished from the edited state, not the original one.")


## Part 5 -- A Real, Measured Comparison

Not an estimate -- an actual line count of the source each approach needed for the identical task, via `inspect.getsource`.

In [ ]:
# `inspect.getsource` isn't reliable against cells executed through nbconvert's
# kernel (no linecache backing "cell" source in this execution mode) -- so this
# measures the exact, literal cell source via IPython's own `In` history instead,
# which IS populated reliably. `In[2]` is the hand-rolled-loop cell above; `In[3]`
# is the graph engine cell; `In[4]` is the node-functions-and-wiring cell.

loop_lines = len(In[2].strip().splitlines())
engine_lines = len(In[3].strip().splitlines())
graph_task_lines = len(In[4].strip().splitlines())

print("-" * 60)
print("REAL, MEASURED COMPARISON (same task, same final answer)")
print("-" * 60)
print(f"{'Approach':<55} {'Lines of code':>15}")
print(f"{'Hand-rolled loop (definition + run, Part 1 cell)':<55} {loop_lines:>15}")
print(f"{'Graph: engine itself (Part 2 cell -- reusable)':<55} {engine_lines:>15}")
print(f"{'Graph: this task on top of the engine (Part 3 cell)':<55} {graph_task_lines:>15}")
print()
print(f"Task-specific logic is comparable ({loop_lines} vs {graph_task_lines} lines) -- the")
print(f"difference is the one-time {engine_lines}-line engine investment, paid once and reused")
print(f"for every future graph, not paid again per task. This is exactly notes Section 9's")
print(f"point: lines of code for THIS ONE task looks close; the real difference shows up in")
print(f"checkpointing, time-travel, and middleware reuse across MANY tasks, which the loop")
print(f"has none of.")


## Optional -- Check for Real LangGraph / Claude Agent SDK

Neither is installed in this environment. This cell checks gracefully rather than assuming, and explains what would change if they were.

In [ ]:
def check_real_frameworks():
    try:
        import langgraph  # noqa: F401
        print("langgraph IS installed -- you could re-run Part 3 against the real library.")
    except ImportError:
        print("langgraph is NOT installed. The mini Graph engine above implements the same")
        print("core semantics (nodes, edges, conditional edges, reducers, checkpoints) as a")
        print("teaching model -- notes Sections 2-3-5 describe what the real library adds on")
        print("top: production-grade persistence backends, streaming, and a far larger node/")
        print("edge/subgraph API surface.")

    try:
        import claude_agent_sdk  # noqa: F401
        print("claude_agent_sdk IS installed -- you could compare Part 1's loop against it directly.")
    except ImportError:
        print("claude_agent_sdk is NOT installed. Part 1's hand-rolled loop is structurally")
        print("what the SDK's own model-driven loop does internally (notes Section 8) --")
        print("the SDK adds built-in tools, hooks, and session management on top.")


check_real_frameworks()


## Key Takeaways

You built a real, working graph engine from scratch -- nodes, fixed and conditional edges, per-field reducers, middleware, and checkpointing -- and watched it reproduce notes Section 11's exact state trace, matching a hand-rolled loop's final answer through a completely different execution shape. Summarization middleware showed Chapter 4's compaction expressed as a reusable wrapper instead of an ad-hoc check; permission middleware blocked and allowed the identical node purely by configuration, with zero change to the node's own code -- notes Section 4's whole argument against reimplementing the same check inside every node. The interrupt-and-resume exercise showed the concrete advantage a checkpointed graph has over a plain loop's in-flight variables: a human could inspect, edit, and resume from *real* state, and that edit provably survived to the final answer.

**Connection forward:** Chapter 12 stays inside durable execution but removes the graph runtime's safety net entirely -- taking Chapter 6's plain loop and building, by hand, the same checkpointing and resume guarantees this chapter's `Graph` class provided for free.